# Brillouin cascade — stochastic SDE in physical Otterstrom units

This notebook is a thin analysis driver for `data/temperature_sweep.json`, produced by
`scripts/sweep_temperature_otterstrom.py`.

The sweep is parameterized by **bath temperature**. Temperatures are edited explicitly in
`TEMPERATURES_K = [...]` inside the sweep script. For each point,

\[
n_{\rm th}(T)=\frac{1}{\exp\!\left(\hbar\Omega_b/k_B T\right)-1},
\qquad \Omega_b=2\pi\times6.02\;\mathrm{GHz}.
\]

All optical modes use the same Stokes decay rate requested for this model,
\(\gamma_j=2\pi\times83\,\mathrm{MHz}\). The phonon decay is
\(\Gamma=2\pi\times13.1\,\mathrm{MHz}\), and \(g=11.1\,\mathrm{kHz}\).

The solver still integrates with its internal coherent-drive amplitude \(E\), but every plot in this
notebook uses **on-chip pump power in mW**. The conversion stored in the JSON is

\[
P_{\rm cav}(E)=\frac{4\hbar\omega_p v_{g,p}}{L\gamma^2}E^2,
\qquad
P_{\rm in}(E)=\frac{P_{\rm cav}(E)}{1.8},
\]

with \(\lambda_p=1535\,\mathrm{nm}\), \(v_{g,p}=7.163\times10^7\,\mathrm{m/s}\),
and \(L=4.576\,\mathrm{cm}\).

## 1. Run the temperature sweep

Edit `TEMPERATURES_K` in `scripts/sweep_temperature_otterstrom.py`, then run from the repository root:

```bash
python scripts/sweep_temperature_otterstrom.py --dry-run --no-log
python scripts/sweep_temperature_otterstrom.py --threads 7
```

The second command writes `data/temperature_sweep.json`.

In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
TEMPERATURE_JSON = DATA / "temperature_sweep.json"

sys.path.insert(0, str(ROOT))
from brillouin import plots as bp
from brillouin import linear_theory as lt
importlib.reload(bp)
importlib.reload(lt)

S = bp.load_temperature_sweep(TEMPERATURE_JSON)
bp.temperature_sweep_report(S)

## 2. Generation curve

SDE mean amplitude of the first Stokes photon mode \(a_2\) versus physical on-chip pump power, one curve per temperature.

In [ ]:
bp.plot_temperature_generation(S, mode=1).show()

## 3. Stationary photon \(g^{(2)}(0)\)

Sub-threshold points with essentially zero photon amplitude are masked by default because their ratio estimator is numerically meaningless.

In [ ]:
bp.plot_temperature_g2(S, mode=1, amp_floor=1e-3).show()

## 4. \(g^{(2)}(0)\) map in the physical \((P_{m in},T)\) plane

In [ ]:
bp.plot_temperature_g2_map(S, mode=1, amp_floor=1e-3).show()

## 5. Apparent generation thresholds versus temperature

The threshold estimator is deliberately simple: the first pump point where the SDE mean amplitude exceeds a fixed fraction of its maximum along that temperature sweep.

In [ ]:
bp.plot_temperature_thresholds(S, frac=0.1).show()

## 6. Phonon thermal-statistics sanity check

For a decoupled thermal complex Gaussian phonon mode, \(g^{(2)}(0)=2\). This plot is retained only as a noise/sampling diagnostic.

In [ ]:
bp.plot_temperature_phonon_g2(S).show()

## 7. Direct theory/conversion sanity checks

`linear_theory.py` is now plot-free. It exposes the same Bose--Einstein and pump-power conversions as the sweep script, plus the exact finite-pump \(N=3\) linearized result.

In [ ]:
m = S["meta"]
gamma = float(m["gamma_opt"])
Gamma = float(m["Gamma"])
g = float(m["g"])

print("n_th(300 K) =", lt.nth_from_temperature(300.0))
print("E2 =", lt.E_threshold2(g, gamma, Gamma), "s^-1")
print("P_in(E2) =", 1e3 * lt.input_power_threshold2(g, gamma, Gamma), "mW")
print("stored JSON P_in(E2) =", 1e3 * float(m["P_input_threshold2_W"]), "mW")